# AXIOM — Faction AI Inference Test Notebook

Run this **after** `train.py` and **before** `ezkl_compile.py`.

Use this to:
- Inspect model predictions on sample game states
- Validate the model makes sensible decisions
- Debug issues before ZK compilation (cheaper to fix here)
- Visualize action distributions across strategies

> **Open in VS Code**: the Jupyter extension renders this directly.
> No Colab needed.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import torch
import torch.nn.functional as F

# Import from train.py
sys.path.insert(0, str(Path(".").resolve()))
from train import (
    ACTION_DIM, ACTION_NAMES, FEATURE_NAMES,
    SEQ_LEN, STATE_DIM, FactionStrategyNet,
    GameDataGenerator,
)

print("✓ Imports OK")
print(f"  State dim  : {STATE_DIM}")
print(f"  Action dim : {ACTION_DIM}")
print(f"  Seq len    : {SEQ_LEN}")
print(f"  Actions    : {ACTION_NAMES}")

## 1. Load Trained Model

In [ ]:
CHECKPOINT = "faction_model.pt"

checkpoint = torch.load(CHECKPOINT, map_location="cpu")
config = checkpoint.get("config", {})

model = FactionStrategyNet(**{k: v for k, v in config.items()})
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"✓ Loaded checkpoint")
print(f"  Epoch     : {checkpoint.get('epoch', '?')}")
print(f"  Val Acc   : {checkpoint.get('val_acc', 0):.4f}")
print(f"  Params    : {sum(p.numel() for p in model.parameters()):,}")

## 2. Single Prediction — Custom Game State

Edit the values below to simulate any scenario.

In [ ]:
# ── Craft a scenario manually ────────────────
# Aggressive faction with high territory, medium threat, good energy

def make_state(
    total_territory=60.0,
    energy_balance=2000.0,
    threat_level=0.4,
    season_progress=0.7,
    attack_power=0.8,
    defense_strength=0.5,
) -> np.ndarray:
    """Create a game state vector with readable parameters."""
    state = np.random.rand(STATE_DIM).astype(np.float32) * 0.1  # small noise base
    state[4]  = total_territory
    state[5]  = energy_balance
    state[9]  = threat_level
    state[11] = season_progress
    state[20] = defense_strength
    state[21] = attack_power
    return state


# Build a sequence of 8 states (slight progression over time)
history = np.stack([
    make_state(total_territory=50 + i*2, season_progress=0.6 + i*0.01)
    for i in range(SEQ_LEN)
])

# Predict
with torch.no_grad():
    input_tensor = torch.from_numpy(history).unsqueeze(0)  # (1, T, D)
    logits = model(input_tensor)
    probs = F.softmax(logits, dim=-1).squeeze().numpy()

predicted_idx = probs.argmax()
print(f"\nPredicted action: [{predicted_idx}] {ACTION_NAMES[predicted_idx]}")
print(f"\nAction probabilities:")
for i, (name, prob) in enumerate(zip(ACTION_NAMES, probs)):
    bar = "█" * int(prob * 30)
    marker = " ← predicted" if i == predicted_idx else ""
    print(f"  {i}. {name:<15} {prob:.4f}  {bar}{marker}")

## 3. Action Distribution Across 3 Strategies

In [ ]:
gen = GameDataGenerator(seed=99)
N = 500
strategies = ["aggressive", "defensive", "balanced"]
colors = ["#F5834C", "#4C8BF5", "#52C97F"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.suptitle("AXIOM — Predicted Action Distribution by Strategy", fontweight="bold", y=1.02)

for ax, strategy, color in zip(axes, strategies, colors):
    counts = np.zeros(ACTION_DIM)

    for _ in range(N):
        history = np.stack([gen._generate_state(strategy) for _ in range(SEQ_LEN)])
        with torch.no_grad():
            t = torch.from_numpy(history).unsqueeze(0)
            logits = model(t)
            action = logits.argmax().item()
        counts[action] += 1

    pct = counts / N * 100
    bars = ax.bar(ACTION_NAMES, pct, color=color, alpha=0.85, edgecolor="white", linewidth=0.5)
    ax.set_title(f"{strategy.capitalize()} faction", fontweight="bold")
    ax.set_xlabel("Action")
    ax.set_xticklabels(ACTION_NAMES, rotation=35, ha="right", fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    ax.set_ylabel("% of predictions" if ax == axes[0] else "")

    # Label top bar
    top = pct.argmax()
    ax.annotate(
        f"{pct[top]:.0f}%",
        xy=(top, pct[top]),
        xytext=(top, pct[top] + 1.5),
        ha="center", fontsize=8, fontweight="bold", color=color
    )

plt.tight_layout()
plt.savefig("action_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: action_distribution.png")

## 4. Validate ONNX vs PyTorch Output

Run this after `export.py` to confirm outputs match.

In [ ]:
ONNX_PATH = "model.onnx"

if not Path(ONNX_PATH).exists():
    print("⚠ model.onnx not found — run export.py first")
else:
    ort_session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
    diffs = []

    for _ in range(50):
        x = torch.randn(1, SEQ_LEN, STATE_DIM)
        with torch.no_grad():
            pt_out = model(x).numpy()
        ort_out = ort_session.run(None, {"game_state_history": x.numpy()})[0]
        diffs.append(np.abs(pt_out - ort_out).max())

    diffs = np.array(diffs)
    print(f"✓ PyTorch vs ONNX comparison (50 samples):")
    print(f"  Max diff   : {diffs.max():.8f}")
    print(f"  Mean diff  : {diffs.mean():.8f}")
    status = "✓ PASS" if diffs.max() < 1e-4 else "⚠ MISMATCH — check model ops"
    print(f"  Status     : {status}")

## 5. Attention Heatmap — What Does the Model Focus On?

In [ ]:
import torch.nn as nn

# Hook to capture attention weights
attention_weights = []

def hook_fn(module, input, output):
    # TransformerEncoderLayer stores attn weights in output[1]
    # We patch to use need_weights=True
    pass

# Build a scenario for visualization
history = np.stack([
    make_state(
        total_territory=10 + i * 8,
        threat_level=0.1 + i * 0.1,
        season_progress=i / SEQ_LEN
    )
    for i in range(SEQ_LEN)
])

# Feature importance via input gradient
input_tensor = torch.from_numpy(history).unsqueeze(0).requires_grad_(True)
logits = model(input_tensor)
predicted = logits.argmax().item()
logits[0, predicted].backward()

# Gradient magnitude per feature
grad = input_tensor.grad.squeeze().abs().mean(0).detach().numpy()  # (STATE_DIM,)
top_features = np.argsort(grad)[::-1][:12]

fig, ax = plt.subplots(figsize=(10, 4))
colors_grad = ["#4C8BF5" if i in top_features[:3] else "#A8C4F5" for i in range(len(top_features))]
ax.barh(
    [FEATURE_NAMES[i] for i in top_features],
    grad[top_features],
    color=colors_grad,
    edgecolor="white",
)
ax.set_title(
    f"Feature Importance for Action: {ACTION_NAMES[predicted]}",
    fontweight="bold"
)
ax.set_xlabel("Gradient Magnitude (importance)")
ax.grid(axis="x", alpha=0.3)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()
print(f"✓ Predicted: [{predicted}] {ACTION_NAMES[predicted]}")
print(f"✓ Top 3 features: {[FEATURE_NAMES[i] for i in top_features[:3]]}")

## 6. Ready for EZKL?

Run this checklist before proceeding to `ezkl_compile.py`.

In [ ]:
checks = [
    ("faction_model.pt exists",    Path("faction_model.pt").exists()),
    ("model_meta.json exists",     Path("model_meta.json").exists()),
    ("model.onnx exists",          Path("model.onnx").exists()),
    ("sample_input.json exists",   Path("sample_input.json").exists()),
    ("Val accuracy > 70%",         checkpoint.get("val_acc", 0) > 0.70),
    ("Param count < 500k",         model.count_parameters() < 500_000),
]

print("EZKL Readiness Checklist")
print("═" * 40)
all_ok = True
for label, result in checks:
    icon = "✓" if result else "✗"
    print(f"  {icon}  {label}")
    if not result:
        all_ok = False

print("═" * 40)
if all_ok:
    print("  ✓ All checks passed — ready for EZKL!")
    print("  Run: python ezkl_compile.py")
else:
    print("  ✗ Fix failing checks before compiling")